In [54]:
!pip install langchain langchain-community pypdf pymupdf sentence-transformers chromadb 

In [55]:
# Correct
from langchain_core.documents import Document

In [56]:
sample_doc = Document(
    page_content="Hello World!", 
    metadata={"source": "https://www.google.com"}
)



In [57]:
sample_doc


Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [58]:
type(sample_doc)

langchain_core.documents.base.Document

In [59]:
# # Text data 
# from langchain_community.document_loaders.text import TextLoader

# loader = TextLoader("data/Python.txt", encoding = "utf-8")

# document = loader.load()

# document

In [60]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/pdf/research.pdf")
# document = pdf_loader.load()
# document

## Ingestion Pipeline

In [61]:
# Data => Document 
import os 
from langchain_community.document_loaders.pdf import PyPDFLoader

In [62]:
def load_all_pdfs():
    folder_path = "Data/pdf"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)
            
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [63]:
all_pdf_documents = load_all_pdfs()


total pdfs: 2
total pages: 32


In [64]:
type(all_pdf_documents[0])

langchain_core.documents.base.Document

## Chunks

In [65]:
# chunks 
# !pip install langchain_text_splitters

In [66]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    
    chunked_docs= text_splitter.split_documents(documents)
    return chunked_docs

In [67]:
chunks = split_docs(all_pdf_documents)

In [68]:
len(chunks)

321

## Embedding

In [69]:
from sentence_transformers import SentenceTransformer

In [70]:
class EmbeddingManager:
    def __init__(self, model_name ="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())
        
    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding shape:", embeddings.shape) # <-- Fixed: changed 'embedding' to 'embeddings'
        return embeddings 

In [71]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


C:\Users\mshiv\AppData\Local\Temp\ipykernel_23352\1659584561.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


## vector Store

In [72]:
import chromadb
import uuid


In [76]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name ="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()
        
    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # Create the Collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embedding in RAG"}     
        )
        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())
        
    def add_documents(self, documents, embeddings):
        # 1. Fixed the comparison logic to check documents vs embeddings
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")

        # 2. FIXED INDENTATION: Moved these out of the 'if' block
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}" 
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        # 3. FIXED POSITION: Moved the database addition OUTSIDE the for loop
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )
        
        print("total documents added in vector store =", len(documents_content))
        print("docs in collection:", self.collection.count())

In [77]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [78]:
# 1 data => documents => chunks => embeddings => store in vector store
texts = [doc.page_content for doc in chunks]
emebedding = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embedding shape: (321, 384)
total documents added in vector store = 321
docs in collection: 321


## Retrieval Pipeline

In [80]:
from sklearn.metrics.pairwise import cosine_similarity

In [101]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # 1. FIXED: Initialize the list right at the start so it's always defined
        retrieved_docs = []
        
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]
        
        # Semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()], 
            n_results=top_k
        )

        # cosine similarity
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadata = results["metadatas"][0]
            distances = results["distances"][0]
            documents_list = results["documents"][0] 
            
            for i, (doc_id, meta, doc_text, distance) in enumerate(zip(ids, metadata, documents_list, distances)):
                similarity_score = 1 - distance 
                
                if similarity_score >= score_threshold:
                    retrieved_docs.append({      
                        "id": doc_id,
                        "document": doc_text,    
                        "metadata": meta,
                        "score": similarity_score,
                        "rank": i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")
                
        else:
            print("no documents found") # (Bonus: fixed typo here from 'dcuments')

        return retrieved_docs

In [102]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [104]:
rag_retriever.retrieve("what is encoder decoder?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_a63665e0-9410-4351-b063-1b07f5f96230',
  'document': 'layers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention',
  'metadata': {'book': 'Advances in Neural Information Processing Systems 30',
   'total_pages': 11,
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'page_label': '3',
   'publisher': 'Curran Associates, Inc.',
   'creator': 'PyPDF',
   'eventtype': 'Poster',
   'firstpage': '5998',
   'creationdate': '2026-06-16T00:27:15+00:00',
   'lastpage': '6008',
   'date': '2017',
   'type': 'Conference Procee